In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn joblib


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

import joblib

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\SHIVAM\OneDrive\Apps\master_full.csv")

df.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,circuitId,name_race,date,driverRef,forename,surname,name_team,status,finished,winner
0,1,18,1,1,22.0,1,1.0,1,1,10.0,...,1,Australian Grand Prix,2008-03-16,hamilton,Lewis,Hamilton,McLaren,Finished,True,1
1,2,18,2,2,3.0,5,2.0,2,2,8.0,...,1,Australian Grand Prix,2008-03-16,heidfeld,Nick,Heidfeld,BMW Sauber,Finished,True,0
2,3,18,3,3,7.0,7,3.0,3,3,6.0,...,1,Australian Grand Prix,2008-03-16,rosberg,Nico,Rosberg,Williams,Finished,True,0
3,4,18,4,4,5.0,11,4.0,4,4,5.0,...,1,Australian Grand Prix,2008-03-16,alonso,Fernando,Alonso,Renault,Finished,True,0
4,5,18,5,1,23.0,3,5.0,5,5,4.0,...,1,Australian Grand Prix,2008-03-16,kovalainen,Heikki,Kovalainen,McLaren,Finished,True,0


In [4]:
print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (26759, 30)
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'name_race', 'date', 'driverRef', 'forename', 'surname', 'name_team', 'status', 'finished', 'winner']


In [5]:
required_columns = [
    "raceId",
    "driverId",
    "constructorId",
    "grid",
    "positionOrder",
    "year",
    "round",
    "circuitId",
    "driverRef",
    "name_team",
    "status"
]

df = df[required_columns].copy()

df.head()

,raceId,driverId,constructorId,grid,positionOrder,year,round,circuitId,driverRef,name_team,status
0,18,1,1,1,1,2008,1,1,hamilton,McLaren,Finished
1,18,2,2,5,2,2008,1,1,heidfeld,BMW Sauber,Finished
2,18,3,3,7,3,2008,1,1,rosberg,Williams,Finished
3,18,4,4,11,4,2008,1,1,alonso,Renault,Finished
4,18,5,1,3,5,2008,1,1,kovalainen,McLaren,Finished


In [6]:
df["top10"] = (
    df["positionOrder"] <= 10
).astype(int)

In [7]:
df[
    [
        "driverRef",
        "grid",
        "positionOrder",
        "top10"
    ]
].head(25)

,driverRef,grid,positionOrder,top10
0,hamilton,1,1,1
1,heidfeld,5,2,1
2,rosberg,7,3,1
3,alonso,11,4,1
4,kovalainen,3,5,1
5,nakajima,13,6,1
6,bourdais,17,7,1
7,raikkonen,15,8,1
8,kubica,2,9,1
9,glock,18,10,1


In [8]:
df["top10"].value_counts()

top10
0    15438
1    11321
Name: count, dtype: int64

In [9]:
df["top10"].value_counts(normalize=True) * 100

top10
0    57.692739
1    42.307261
Name: proportion, dtype: float64

In [10]:
df = df.sort_values(
    by=["year", "round", "raceId"]
).reset_index(drop=True)

In [11]:
df[
    ["year", "round", "raceId", "driverRef"]
].head()

,year,round,raceId,driverRef
0,1950,1,833,farina
1,1950,1,833,fagioli
2,1950,1,833,reg_parnell
3,1950,1,833,cabantous
4,1950,1,833,rosier


In [12]:
df["grid"].value_counts().sort_index().head()

grid
0    1638
1    1136
2    1125
3    1130
4    1132
Name: count, dtype: int64

In [13]:
df["grid"] = df["grid"].replace(0, np.nan)

In [14]:
df["previous_finish"] = (
    df.groupby("driverId")["positionOrder"]
      .shift(1)
)

In [15]:
df[
    df["driverRef"] == "hamilton"
][
    [
        "year",
        "round",
        "positionOrder",
        "previous_finish"
    ]
].head(10)

,year,round,positionOrder,previous_finish
19243,2007,1,3,NaN
19264,2007,2,2,3.0
19286,2007,3,2,2.0
19308,2007,4,2,2.0
19330,2007,5,2,2.0
19351,2007,6,1,2.0
19373,2007,7,1,1.0
19397,2007,8,3,1.0
19419,2007,9,3,3.0
19447,2007,10,9,3.0


In [16]:
df["recent_avg_finish"] = (
    df.groupby("driverId")["positionOrder"]
      .transform(
          lambda x: x.shift(1)
                     .rolling(
                         window=5,
                         min_periods=1
                     )
                     .mean()
      )
)

In [17]:
df["recent_top10_rate"] = (
    df.groupby("driverId")["top10"]
      .transform(
          lambda x: x.shift(1)
                     .rolling(
                         window=5,
                         min_periods=1
                     )
                     .mean()
      )
)

In [18]:
df["recent_avg_grid"] = (
    df.groupby("driverId")["grid"]
      .transform(
          lambda x: x.shift(1)
                     .rolling(
                         window=5,
                         min_periods=1
                     )
                     .mean()
      )
)

In [19]:
df["driver_circuit_avg_finish"] = (
    df.groupby(
        ["driverId", "circuitId"]
    )["positionOrder"]
    .transform(
        lambda x: x.shift(1)
                   .expanding()
                   .mean()
    )
)

In [20]:
constructor_race_form = (
    df.groupby(
        ["raceId", "constructorId"],
        as_index=False
    )["top10"]
    .mean()
    .rename(
        columns={
            "top10": "constructor_race_top10_rate"
        }
    )
)

In [21]:
race_information = (
    df[
        ["raceId", "year", "round"]
    ]
    .drop_duplicates()
)

constructor_race_form = constructor_race_form.merge(
    race_information,
    on="raceId",
    how="left"
)

In [22]:
constructor_race_form = (
    constructor_race_form
    .sort_values(
        ["year", "round", "raceId"]
    )
)

In [23]:
constructor_race_form[
    "constructor_recent_top10_rate"
] = (
    constructor_race_form
    .groupby("constructorId")[
        "constructor_race_top10_rate"
    ]
    .transform(
        lambda x: x.shift(1)
                   .rolling(
                       window=5,
                       min_periods=1
                   )
                   .mean()
    )
)

In [24]:
df = df.merge(
    constructor_race_form[
        [
            "raceId",
            "constructorId",
            "constructor_recent_top10_rate"
        ]
    ],
    on=["raceId", "constructorId"],
    how="left"
)

In [25]:
model_columns = [
    "year",
    "round",
    "raceId",
    "driverId",
    "constructorId",
    "circuitId",
    "grid",
    "previous_finish",
    "recent_avg_finish",
    "recent_top10_rate",
    "recent_avg_grid",
    "driver_circuit_avg_finish",
    "constructor_recent_top10_rate",
    "positionOrder",
    "top10"
]

display(df[model_columns].head(30))

,year,round,raceId,driverId,constructorId,circuitId,grid,previous_finish,recent_avg_finish,recent_top10_rate,recent_avg_grid,driver_circuit_avg_finish,constructor_recent_top10_rate,positionOrder,top10
0,1950,1,833,642,51,9,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1,1
1,1950,1,833,786,51,9,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2,1
2,1950,1,833,686,51,9,4.0,NaN,NaN,NaN,NaN,NaN,NaN,3,1
3,1950,1,833,704,154,9,6.0,NaN,NaN,NaN,NaN,NaN,NaN,4,1
4,1950,1,833,627,154,9,9.0,NaN,NaN,NaN,NaN,NaN,NaN,5,1
5,1950,1,833,619,151,9,13.0,NaN,NaN,NaN,NaN,NaN,NaN,6,1
6,1950,1,833,787,151,9,15.0,NaN,NaN,NaN,NaN,NaN,NaN,7,1
7,1950,1,833,741,154,9,14.0,NaN,NaN,NaN,NaN,NaN,NaN,8,1
8,1950,1,833,784,105,9,16.0,NaN,NaN,NaN,NaN,NaN,NaN,9,1
9,1950,1,833,778,105,9,20.0,NaN,NaN,NaN,NaN,NaN,NaN,10,1


In [26]:
df[model_columns].isnull().sum()

year                                0
round                               0
raceId                              0
driverId                            0
constructorId                       0
circuitId                           0
grid                             1638
previous_finish                   861
recent_avg_finish                 861
recent_top10_rate                 861
recent_avg_grid                  1281
driver_circuit_avg_finish        8176
constructor_recent_top10_rate     348
positionOrder                       0
top10                               0
dtype: int64

In [27]:
leakage_columns = [
    "position",
    "positionText",
    "positionOrder",
    "points",
    "laps",
    "time",
    "milliseconds",
    "fastestLap",
    "rank",
    "fastestLapTime",
    "fastestLapSpeed",
    "statusId",
    "status",
    "finished",
    "winner"
]

In [28]:
features = [
    "year",
    "round",
    "grid",
    "driverId",
    "constructorId",
    "circuitId",
    "previous_finish",
    "recent_avg_finish",
    "recent_top10_rate",
    "recent_avg_grid",
    "driver_circuit_avg_finish",
    "constructor_recent_top10_rate"
]

X = df[features].copy()
y = df["top10"].copy()

In [29]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())

Feature shape: (26759, 12)
Target shape: (26759,)


,year,round,grid,driverId,constructorId,circuitId,previous_finish,recent_avg_finish,recent_top10_rate,recent_avg_grid,driver_circuit_avg_finish,constructor_recent_top10_rate
0,1950,1,1.0,642,51,9,NaN,NaN,NaN,NaN,NaN,NaN
1,1950,1,2.0,786,51,9,NaN,NaN,NaN,NaN,NaN,NaN
2,1950,1,4.0,686,51,9,NaN,NaN,NaN,NaN,NaN,NaN
3,1950,1,6.0,704,154,9,NaN,NaN,NaN,NaN,NaN,NaN
4,1950,1,9.0,627,154,9,NaN,NaN,NaN,NaN,NaN,NaN
